# HachimiMT Benchmark Profile

Notebook này dùng để đo tốc độ dịch file và in dòng `BENCH_PROFILE` để soi bottleneck CPU/GPU.

- Chạy được trên Google Colab và Kaggle.
- Tự tải bản code mới nhất từ HF Space `hachimimt-local.zip`.
- Colab: cell chọn file sẽ mở upload nếu bạn chưa set `INPUT_PATH`.
- Kaggle: thêm file `.txt` bằng **Add Input** hoặc đặt file trong `/kaggle/working`, rồi chạy cell chọn file.

Dòng cần xem nằm gần cuối output cell benchmark:

```text
BENCH_RUNTIME ... ct2_cuda_devices=...
BENCH_PACKAGES ... ctranslate2=... sentencepiece=... torch=...
BENCH_ENV ... HACHIMIMT_GPU_INDICES=... HACHIMIMT_CT2_WINDOW_MULTIPLIER=...
BENCH_PROFILE ... chunk_s=... ct2_infer_s=... decode_s=... tokenize_wait_s=...
BENCH_DONE ...
```

Preset hiện tại dùng để đo cấu hình đã chốt trên Kaggle T4 x2: chạy cả 2 GPU, batch `96`, `batch_type=tokens`, window `8` (effective 32x), beam `2`. Nếu muốn kiểm chứng lại, bật cell sweep để chạy window `[4, 8, 16]`. Để so Kaggle với Colab công bằng, bật `USE_SINGLE_GPU_FOR_FAIR_TEST=True`; notebook sẽ tự dùng window `16` cho lần fair test nếu bạn không override `WINDOW_MULTIPLIER`.

In [ ]:
# 1. Tải code mới + cài dependencies
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = Path("/content").exists() and not IN_KAGGLE
WORKDIR = Path("/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else ".").resolve()
os.chdir(WORKDIR)

ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
zip_path = WORKDIR / "hachimimt-local.zip"

print("Runtime:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "Local")
print("Working dir:", WORKDIR)
print("Downloading:", ZIP_URL)
urllib.request.urlretrieve(ZIP_URL, zip_path)

shutil.rmtree(WORKDIR / "hachimimt", ignore_errors=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(WORKDIR)

print("Extracted:", sorted(p.name for p in (WORKDIR / "hachimimt").iterdir()))
print("Installing requirements...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(WORKDIR / "hachimimt" / "requirements.txt")])
print("Setup done")

In [ ]:
# 2. Chọn file input
# - Colab: để trống INPUT_PATH thì notebook sẽ mở hộp upload.
# - Kaggle: dùng Add Input để gắn dataset .txt, hoặc điền path cụ thể dưới đây.
from pathlib import Path
import sys

INPUT_PATH = ""  # ví dụ Kaggle: "/kaggle/input/my-dataset/book.txt"; Colab: "/content/book.txt"

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def resolve_input_path() -> Path:
    raw = INPUT_PATH.strip()
    if raw:
        path = Path(raw).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"INPUT_PATH không tồn tại: {path}")
        return path

    if _is_colab_runtime():
        from google.colab import files  # type: ignore
        print("Chọn/upload file .txt từ máy của bạn...")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("Bạn chưa upload file nào.")
        name = next(iter(uploaded))
        return Path(name).resolve()

    candidates = []
    for root in ["/kaggle/input", "/kaggle/working", "/content"]:
        base = Path(root)
        if base.exists():
            candidates.extend(p for p in base.rglob("*.txt") if p.is_file())

    candidates = sorted(set(candidates), key=lambda p: p.stat().st_size, reverse=True)
    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy file .txt. Trên Kaggle: bấm Add Input để gắn dataset chứa .txt, "
            "hoặc upload/tạo file trong /kaggle/working rồi chạy lại cell này."
        )

    print("Tìm thấy file .txt, chọn file lớn nhất:")
    for idx, path in enumerate(candidates[:10], start=1):
        print(f"{idx}. {path} ({path.stat().st_size:,} bytes)")
    return candidates[0]

input_path = resolve_input_path()
print("INPUT_FILE=", input_path)
print("size_bytes=", input_path.stat().st_size)

In [ ]:
# 3. Cấu hình benchmark
# Đổi các biến ở đây rồi chạy lại cell benchmark bên dưới.
import os

MODEL = "HachimiMT-60"      # HachimiMT-60, HachimiMT-30, MoxhiMT-60, MoxhiMT-30, HirashibaMT-Medium, HirashibaMT-Tiny
BEAM = 2                    # 1 nhanh hơn, 2 thường cân bằng hơn
CHUNK_MODE = "sentence"     # sentence hoặc paragraph
NORMALIZE = "auto"          # auto, t2s, none
PROGRESS_SECONDS = 30

# Preset hiện tại ưu tiên benchmark đơn với cấu hình đã chốt.
# Đổi thành True nếu bạn muốn chạy một cấu hình đơn trước sweep.
RUN_SINGLE_BENCHMARK = True

# Bật để so Kaggle x1 T4 công bằng với Colab x1 T4.
# Tắt để Kaggle tự dùng toàn bộ GPU được cấp, ví dụ T4 x2.
USE_SINGLE_GPU_FOR_FAIR_TEST = False
FAIR_GPU_INDICES = "0"

# Các giá trị này sẽ truyền vào subprocess benchmark trước khi app import CT2.
# Để "" nếu muốn dùng auto/default của app.
BATCH_SIZE = "96"
WINDOW_MULTIPLIER = "8"      # Kaggle T4 x2: 8 nhanh nhất trong sweep; Colab fair x1 dùng 16
FAIR_WINDOW_MULTIPLIER = "16"
CT2_BATCH_TYPE = "tokens"
INTER_THREADS = "1"
TOKENIZE_WORKERS = ""      # Colab 2 vCPU có thể thử "2"; để trống = auto
TOKENIZE_JOB_SIZE = ""
CT2_THREADS = ""

TRACKED_ENV_KEYS = [
    "CUDA_VISIBLE_DEVICES",
    "HACHIMIMT_GPU_INDICES",
    "HACHIMIMT_AUTO_ALL_GPUS",
    "HACHIMIMT_BATCH_SIZE",
    "HACHIMIMT_THREADS",
    "HACHIMIMT_TOKENIZE_WORKERS",
    "HACHIMIMT_TOKENIZE_JOB_SIZE",
    "HACHIMIMT_CT2_BATCH_TYPE",
    "HACHIMIMT_CT2_WINDOW_MULTIPLIER",
    "HACHIMIMT_INTER_THREADS",
]


def _set_or_unset(env, key, value):
    value = str(value).strip()
    if value:
        env[key] = value
    else:
        env.pop(key, None)


def build_benchmark_env(
    *,
    use_single_gpu=USE_SINGLE_GPU_FOR_FAIR_TEST,
    gpu_indices=FAIR_GPU_INDICES,
    batch_size=BATCH_SIZE,
    window_multiplier=WINDOW_MULTIPLIER,
    fair_window_multiplier=FAIR_WINDOW_MULTIPLIER,
    ct2_batch_type=CT2_BATCH_TYPE,
    inter_threads=INTER_THREADS,
    tokenize_workers=TOKENIZE_WORKERS,
    tokenize_job_size=TOKENIZE_JOB_SIZE,
    ct2_threads=CT2_THREADS,
):
    env = os.environ.copy()
    if use_single_gpu:
        env["HACHIMIMT_GPU_INDICES"] = str(gpu_indices).strip() or "0"
        env["HACHIMIMT_AUTO_ALL_GPUS"] = "0"
        if not str(window_multiplier).strip():
            window_multiplier = fair_window_multiplier
    else:
        env.pop("HACHIMIMT_GPU_INDICES", None)
        env.pop("HACHIMIMT_AUTO_ALL_GPUS", None)

    _set_or_unset(env, "HACHIMIMT_BATCH_SIZE", batch_size)
    _set_or_unset(env, "HACHIMIMT_CT2_WINDOW_MULTIPLIER", window_multiplier)
    _set_or_unset(env, "HACHIMIMT_CT2_BATCH_TYPE", ct2_batch_type)
    _set_or_unset(env, "HACHIMIMT_INTER_THREADS", inter_threads)
    _set_or_unset(env, "HACHIMIMT_TOKENIZE_WORKERS", tokenize_workers)
    _set_or_unset(env, "HACHIMIMT_TOKENIZE_JOB_SIZE", tokenize_job_size)
    _set_or_unset(env, "HACHIMIMT_THREADS", ct2_threads)
    return env


preview_env = build_benchmark_env()
print("Benchmark config:")
print("MODEL=", MODEL, "BEAM=", BEAM, "CHUNK_MODE=", CHUNK_MODE, "NORMALIZE=", NORMALIZE)
for key in TRACKED_ENV_KEYS:
    if key in preview_env:
        print(f"{key}={preview_env[key]}")
print("single_gpu_fair_test=", USE_SINGLE_GPU_FOR_FAIR_TEST)

In [ ]:
# 4. Chạy benchmark và in BENCH_PROFILE
import os
import subprocess
import sys
from pathlib import Path

bench_script = Path("hachimimt/src/benchmark_file.py")
if not bench_script.exists():
    raise FileNotFoundError(f"Không thấy benchmark script: {bench_script}")
if not Path(input_path).exists():
    raise FileNotFoundError(f"Input file không tồn tại: {input_path}")


def parse_kv_line(line):
    data = {}
    for part in line.split()[1:]:
        if "=" in part:
            key, value = part.split("=", 1)
            data[key] = value
    return data


def run_benchmark_once(
    *,
    label="single",
    model=MODEL,
    beam=BEAM,
    chunk_mode=CHUNK_MODE,
    normalize=NORMALIZE,
    progress_seconds=PROGRESS_SECONDS,
    use_single_gpu=USE_SINGLE_GPU_FOR_FAIR_TEST,
    gpu_indices=FAIR_GPU_INDICES,
    batch_size=BATCH_SIZE,
    window_multiplier=WINDOW_MULTIPLIER,
):
    env = build_benchmark_env(
        use_single_gpu=use_single_gpu,
        gpu_indices=gpu_indices,
        batch_size=batch_size,
        window_multiplier=window_multiplier,
    )
    cmd = [
        sys.executable,
        str(bench_script),
        str(input_path),
        "--model", model,
        "--backend", "ct2",
        "--beam", str(beam),
        "--chunk-mode", chunk_mode,
        "--normalize", normalize,
        "--progress-seconds", str(progress_seconds),
    ]

    print(f"RUN_LABEL={label}")
    print("RUN:", " ".join(cmd))
    print("ENV:", {key: env.get(key) for key in TRACKED_ENV_KEYS if env.get(key) is not None})
    print("\n--- benchmark output ---")
    lines = []
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        lines.append(line.rstrip("\n"))
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)

    profile_lines = [line for line in lines if line.startswith("BENCH_PROFILE")]
    done_lines = [line for line in lines if line.startswith("BENCH_DONE")]
    runtime_lines = [line for line in lines if line.startswith("BENCH_RUNTIME")]
    package_lines = [line for line in lines if line.startswith("BENCH_PACKAGES")]
    env_lines = [line for line in lines if line.startswith("BENCH_ENV")]
    summary = {
        "label": label,
        "profile_line": profile_lines[-1] if profile_lines else "",
        "done_line": done_lines[-1] if done_lines else "",
        "runtime_line": runtime_lines[-1] if runtime_lines else "",
        "package_line": package_lines[-1] if package_lines else "",
        "env_line": env_lines[-1] if env_lines else "",
        "profile": parse_kv_line(profile_lines[-1]) if profile_lines else {},
        "done": parse_kv_line(done_lines[-1]) if done_lines else {},
    }
    print("\n--- parsed summary ---")
    print(summary["runtime_line"] or "Không thấy BENCH_RUNTIME. Hãy chắc notebook đã tải zip mới từ Space.")
    print(summary["package_line"] or "Không thấy BENCH_PACKAGES.")
    print(summary["env_line"] or "Không thấy BENCH_ENV.")
    print(summary["profile_line"] or "Không thấy BENCH_PROFILE.")
    print(summary["done_line"] or "Không thấy BENCH_DONE.")
    return summary


if RUN_SINGLE_BENCHMARK:
    last_summary = run_benchmark_once()
else:
    print("RUN_SINGLE_BENCHMARK=False. Bỏ qua benchmark đơn, chạy tiếp cell sweep.")

In [ ]:
# 5. Sweep batch/window/beam bằng script CLI
# Preset tuỳ chọn: kiểm chứng window trên Kaggle T4 x2 sau khi batch=96 đã chốt.
RUN_SWEEP = False
SWEEP_SINGLE_GPU = False     # False: Kaggle dùng auto/all GPU; True: fair x1 T4
SWEEP_BEAMS = [1, 2]
SWEEP_BATCHES = [96]
SWEEP_WINDOWS = [4, 8, 16]   # Kaggle x2: window 8/16 đều đạt effective 32x; 8 đang nhỉnh nhất
SWEEP_PROGRESS_SECONDS = 999999
SWEEP_MAX_RUNS = 0           # 0 = chạy hết combo; đặt 1/2 để smoke nhanh

def _csv(values):
    return ",".join(str(value) for value in values)


if RUN_SWEEP:
    sweep_script = Path("hachimimt/src/benchmark_sweep.py")
    if not sweep_script.exists():
        raise FileNotFoundError(f"Không thấy sweep script: {sweep_script}")

    cmd = [
        sys.executable,
        str(sweep_script),
        str(input_path),
        "--model", MODEL,
        "--backend", "ct2",
        "--chunk-mode", CHUNK_MODE,
        "--normalize", NORMALIZE,
        "--beams", _csv(SWEEP_BEAMS),
        "--batches", _csv(SWEEP_BATCHES),
        "--windows", _csv(SWEEP_WINDOWS),
        "--ct2-batch-type", CT2_BATCH_TYPE,
        "--inter-threads", INTER_THREADS,
        "--progress-seconds", str(SWEEP_PROGRESS_SECONDS),
    ]
    if SWEEP_SINGLE_GPU:
        cmd.extend(["--single-gpu", "--gpu-indices", FAIR_GPU_INDICES])
    else:
        cmd.append("--auto-all-gpus")
    if TOKENIZE_WORKERS:
        cmd.extend(["--tokenize-workers", TOKENIZE_WORKERS])
    if TOKENIZE_JOB_SIZE:
        cmd.extend(["--tokenize-job-size", TOKENIZE_JOB_SIZE])
    if CT2_THREADS:
        cmd.extend(["--ct2-threads", CT2_THREADS])
    if SWEEP_MAX_RUNS:
        cmd.extend(["--max-runs", str(SWEEP_MAX_RUNS)])

    print("RUN_SWEEP_CMD:", " ".join(cmd))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
else:
    print("RUN_SWEEP=False. Đổi thành True để chạy mini-sweep batch/window/beam.")

## Đọc kết quả

- `ct2_infer_s`: thời gian inference thật trong CTranslate2. Nếu dòng này chiếm phần lớn, bottleneck chính là model/GPU.
- `chunk_s`: thời gian chia chunk và đếm token.
- `decode_s`: thời gian decode output.
- `tokenize_wait_s`: thời gian GPU phải chờ tokenization. Nếu cao trên Colab/Kaggle, CPU đang nghẽn.
- `BENCH_RUNTIME`: Python/platform và số GPU CT2 nhìn thấy.
- `BENCH_PACKAGES`: version package quan trọng; khác version CT2/SentencePiece có thể làm lệch tốc độ.
- `BENCH_ENV`: cấu hình hiệu năng thật được truyền vào subprocess benchmark.
- `BENCH_DONE chars_s`: tốc độ chữ Hán/giây tính trên thời gian dịch.
- Fair test Kaggle vs Colab: đặt `USE_SINGLE_GPU_FOR_FAIR_TEST=True`, `FAIR_GPU_INDICES="0"`; nếu `WINDOW_MULTIPLIER=""`, notebook tự dùng `FAIR_WINDOW_MULTIPLIER="16"`.